In [ ]:
%pip install ucimlrepo lime scikit-learn numpy
from ucimlrepo import fetch_ucirepo 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from lime import lime_tabular
from IPython.display import HTML, display
import time
  
# fetch dataset 
covertype = fetch_ucirepo(id=31) 
  
# data (as pandas dataframes) 
X = covertype.data.features #ex: slope, aspect, elevation
y = covertype.data.targets.squeeze() #ex: forest cover type (1-7)
#sqqueeze so y is series instead of dataframe - makes it easier to work with in sklearn
  
# metadata 
print("Covertype metadata:", covertype.metadata) 
  
# variable information 
print("Covertype variables: ", covertype.variables) 

#load dataset 
#split into train and test sets
# display description of dataset and it's features
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42, # makes split reproducible otherwise train/test split could change each time
    stratify=y # tries to maintain approx same proportion of each forest cover class in training and test dataset
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("First few rows: ", X.head())
print("First few rows: ", y.head())

# 2. Train a complex "black box" model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train) # trains model - what covertype likely?


# 3. Check the accuracy on the model on test data

# Make predictions for the test set - model never trained on this
y_pred_test = model.predict(X_test)
#predictions stored in y_pred_test


# View accuracy score - compares actual labels and predicted labels - so if 0.85 returns - 85% classified correctly
print("Accuracy score: ", accuracy_score(y_test, y_pred_test))

# classification report
# View the classification report for test data and predictions
print("Classification report: ", classification_report(y_test, y_pred_test))
#gives you metrics such as:
#   precision
#   recall
#   F1-score
#   support
#   
# for each cover-type class.


# 4. Initialize the LIME Tabular Explainer
feature_names = X.columns.tolist()
# Use the model's actual class order to keep LIME labels consistent
class_names = [str(c) for c in model.classes_]

start = time.perf_counter() # start of runtime measurement

explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=feature_names,
    class_names=class_names,
    mode="classification"
)

runtime = time.perf_counter() - start

print("Runtime:", runtime)

# 5. Pick an instance from the test set to explain
instance_idx = 0
instance = X_test.iloc[instance_idx]  # iloc means select based on row position - so first row of test set
true_class = y_test.iloc[instance_idx]
instance_df = instance.to_frame().T
predicted_class = model.predict(instance_df)[0]

# Probability for every class
pred_proba = model.predict_proba(instance_df)[0]
# gives probability of each class for the instance - returns a list of probabilities for each class

print(f"True label: {true_class}")
print(f"Model prediction: {predicted_class}")  # class with largest probability is the predicted class

# Get the exact class index used by the model
predicted_class_index = int(np.where(model.classes_ == predicted_class)[0][0])

# 6. Generate the local explanation - "Which features are most responsible for this particular prediction?"
exp = explainer.explain_instance(
    data_row=instance.values,
    predict_fn=model.predict_proba,
    labels=(predicted_class_index,),
    num_features=5,
    num_samples=250
)

# Use the actual label that exists in exp.local_exp to avoid KeyError
valid_label = next(iter(exp.local_exp.keys()))

# num_features = take 5 most important features for this instance - can change to 10 or 20 if you want more features

# Display feature weights in terminal
print("\n--- LIME Feature Contributions ---")
for feature, weight in exp.as_list(label=valid_label):
    direction = "Supports" if weight > 0 else "Opposes"
    print(f"{feature:<40} | Weight: {weight:+.4f} ({direction})")

In [ ]:
# In a Jupyter notebook, render the interactive visualization:
display(HTML(exp.as_html(show_table=True)))